<a href="https://colab.research.google.com/github/prasadboi/llm-from-scratch/blob/main/tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tokenization
- Steps
  - split text into words and subwords called tokens
  - tokens are converted into token ids
  - token IDs are mapped to vectors

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## General Idea and examples on how tokenization works using regex examples

In [28]:
# Creating the tokens
## Downloading the dataset (The Verdict by Edith Wharton)
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "/content/drive/MyDrive/Project_Work/LLMFromScratch/the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

## Peeking into the data
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"total number of characters in the book are: {len(raw_text)}")
print(raw_text[:99])

## Now we wish to tokenize this file / text-string
### Note that this implementation is not a scalable one like the ones that huggingface tokenizers uses.
### The reason being that we are doing for 1 text file only.

## splitting using regex for this implementation
import re
### Whether we should include whitespaces or not?
#### Requirements dependent
##### Adv of removing: removing whitespaces reduces the mem and computing requirements. and we can insert manually during token generation
##### Adv of including: useful when the whitespaces affect the structure of the text. Eg. indentation in Python.

### For this text, to find out what special characters are there beyond whitespaces, i can use a regex command and get those occurrences
spl_characters = re.findall(r'[^A-Z^a-z^0-9\s]', raw_text)
spl_characters = list(dict.fromkeys(spl_characters))
print(f"spl_characters in this text are: {spl_characters}")
split_token_regex = fr'(--|[{"".join(re.escape(c) for c in spl_characters)}]|\s)'
print(f"regex for tokenization is: {split_token_regex}")
tokenized_text = [item.strip() for item in re.split(split_token_regex, raw_text) if item.strip()]
print(f"total number of tokens are: {len(tokenized_text)}")
print(f"sample tokens: {tokenized_text[:40]}")
spl_character_count = sum(1 for t in tokenized_text if t in spl_characters)
print(f"total number of special characters are: {spl_character_count}")

total number of characters in the book are: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 
spl_characters in this text are: ['-', ',', '.', '(', ')', '"', "'", ';', '_', ':', '?', '!']
regex for tokenization is: (--|[\-,\.\(\)"';_:\?!]|\s)
total number of tokens are: 4766
sample tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his']
total number of special characters are: 843


## Converting the tokens to token ids

In [19]:
# How to do this?
## First build a vocabulary - list of tokens sorted in a particular manner - say alphabetical order
## Each unique token is mapped has its index as its token id. Therefore each token has its own unique token id. (which is the invariant that we need)
vocabulary = sorted(set(tokenized_text))
vocabulary_len = len(vocabulary)
### printing what this dictionary looks like
for i, token in enumerate(vocabulary):
  print(f"{i}: {token}")
  if i >= 49:
    break
### You can think of this mapping as encoding and therefore can be thought of as what the encoder is doing
### Therefore, we now need the decoder (to map these IDs back to words)
#### Here the implementation is a list but ideal way is to think you have a mapping for the encoder and then you have an inverse mapping for the decoder

0: !
1: "
2: '
3: (
4: )
5: ,
6: -
7: --
8: .
9: :
10: ;
11: ?
12: A
13: Ah
14: Among
15: And
16: Are
17: Arrt
18: As
19: At
20: Be
21: Begin
22: Burlington
23: But
24: By
25: Carlo
26: Chicago
27: Claude
28: Come
29: Croft
30: Destroyed
31: Devonshire
32: Don
33: Dubarry
34: Emperors
35: Florence
36: For
37: Gallery
38: Gideon
39: Gisburn
40: Gisburns
41: Grafton
42: Greek
43: Grindle
44: Grindles
45: HAD
46: Had
47: Hang
48: Has
49: He


## Now let us implement the tokenizer class in python

In [69]:
class Tokenizer:
    def __init__(self, vocabulary):
        self.encoder_map = vocabulary
        self.decoder_map = {idx : token for idx, token in enumerate(vocabulary)}
        self.vocabulary_len = len(vocabulary)

    def encode(self, text):
        preprocessed_text = re.split(r'(--|[;_:\?!-,."()\']|\s)', text)
        preprocessed_text = [t.strip() for t in preprocessed_text if t.strip()]
        return [self.encoder_map.index(token) for token in preprocessed_text]

    def decode(self, token_ids):
        decoded_text = " ".join([self.decoder_map[token_id] for token_id in token_ids])
        decoded_text = re.sub(r'\s+(--|[;_:\?!-,."()])', r'\1', decoded_text)
        decoded_text = re.sub(r'\s+\'', '\1', decoded_text)
        return decoded_text

In [73]:
tokenizer = Tokenizer(vocabulary)
text = "It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."
tokenized_ids = tokenizer.encode(text)
print(f"tokenized text: {tokenized_ids}")
decoded_text = tokenizer.decode(tokenized_ids)
print(f"decoded text: {decoded_text}")

### NOTE: the limitation of the decoder and encoder would be that it is limited by the size of the vocabulary
### therefore it calls for the need to use large and diverse training sets to extend the vocabulary

### LLMs like chat gpt use SPL CONTEXT TOKENS to deal with words that are not present in the vocabulary

tokenized text: [1156, 57, 2, 859, 998, 606, 536, 751, 5, 1136, 600, 5, 68, 8, 39, 860, 1118, 760, 801, 8, 1157]
decoded text: |<sos>| It's the last he painted, you know, Mrs. Gisburn said with pardonable pride. |<eos>|


In [72]:
# Dealing with the spl context tokens
## We shall modify the tokenizer to handle unkown tokens
class Tokenizer:
    def __init__(self, vocabulary):
        vocabulary.append("|<unk>|")
        vocabulary.append("|<sos>|")
        vocabulary.append("|<eos>|")
        self.encoder_map = vocabulary
        self.decoder_map = {idx : token for idx, token in enumerate(vocabulary)}
        self.vocabulary_len = len(vocabulary)

    def encode(self, text):
        preprocessed_text = re.split(r'(--|[;_:\?!-,."()\']|\s)', text)
        preprocessed_text = [t.strip() for t in preprocessed_text if t.strip()]
        preprocessed_text = [token if token in vocabulary else "|<unk>|" for token in preprocessed_text]
        preprocessed_text.insert(0, "|<sos>|")
        preprocessed_text.append("|<eos>|")
        return [self.encoder_map.index(token) for token in preprocessed_text]

    def decode(self, token_ids):
        decoded_text = " ".join([self.decoder_map[token_id] for token_id in token_ids])
        decoded_text = re.sub(r'\s+(--|[;_:\?!-,."()])', r'\1', decoded_text)
        decoded_text = re.sub(r"\s+'", "'", decoded_text)
        decoded_text = re.sub(r"'\s+", "'", decoded_text)
        return decoded_text

In [79]:
tokenizer = Tokenizer(vocabulary)
text1 = "It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."
text2 = "Hello, do you like tea?"
text = text1 + "|<eos>|" + text2
tokenized_ids = tokenizer.encode(text)
print(f"tokenized text: {tokenized_ids}")
decoded_text = tokenizer.decode(tokenized_ids)
print(f"decoded text: {decoded_text}")

tokenized text: [1156, 57, 2, 859, 998, 606, 536, 751, 5, 1136, 600, 5, 68, 8, 39, 860, 1118, 760, 801, 8, 1155, 5, 362, 1136, 631, 985, 11, 1157]
decoded text: |<sos>| It's the last he painted, you know, Mrs. Gisburn said with pardonable pride. |<unk>|, do you like tea? |<eos>|
